In [15]:
# All required imports
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler, MinMaxScaler
import torch.nn as nn
import torch.nn.functional as F
import random
from collections import Counter, defaultdict
import os
import time
import json
import psutil
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
import matplotlib.pyplot as plt
from tqdm import tqdm

In [16]:
import torch

# Check if GPU is available, otherwise use CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🔹 Using device: {device}")
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
import torch.nn as nn
import torch.nn.functional as F
import random
from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import LogisticRegression
from collections import Counter
from sklearn.metrics import roc_auc_score


import torch
import torch.nn as nn
import torch.nn.functional as F

import torch
import torch.nn as nn
import torch.nn.functional as F

class XPA(nn.Module):
    def __init__(self, input_dim, position_dim, hidden_dim=64, attention_dim=32):
        super(XPA, self).__init__()

        # Embeddings for position
        self.position_embedding = nn.Linear(1, position_dim)

        # Project input features
        self.doc_proj = nn.Linear(input_dim, attention_dim)
        self.pos_proj = nn.Linear(position_dim, attention_dim)

        # Attention components
        self.attention_query = nn.Linear(attention_dim, attention_dim)
        self.attention_key = nn.Linear(attention_dim, attention_dim)
        self.attention_value = nn.Linear(attention_dim, attention_dim)

        # Final tower
        self.mlp = nn.Sequential(
            nn.Linear(attention_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1),
            nn.Sigmoid()
        )

    def forward(self, x, pos):
        """
        x: [B, input_dim]
        pos: [B, 1] (float position index)
        """
        # Step 1: Encode position
        pos_emb = self.position_embedding(pos)  # [B, position_dim]

        # Step 2: Project both into attention space
        doc_vec = self.doc_proj(x)              # [B, attention_dim]
        pos_vec = self.pos_proj(pos_emb)        # [B, attention_dim]

        # Step 3: Cross-Positional Attention
        Q = self.attention_query(doc_vec)       # [B, attention_dim]
        K = self.attention_key(pos_vec)         # [B, attention_dim]
        V = self.attention_value(pos_vec)       # [B, attention_dim]

        # Attention: single-head additive attention
        attn_scores = torch.tanh(Q + K)         # [B, attention_dim]
        attended = attn_scores * V              # Element-wise attention

        # Step 4: Final tower
        out = self.mlp(attended)                # [B, 1]
        return out

input_dim = 700          # main feature dimension from Yahoo LTR
position_dim = 1         # position is 1D
bias_dim = 1             # bias is 1D

model = XPA(input_dim=input_dim, position_dim=position_dim).to(device)



class YahooLTRSet2Dataset(Dataset):
    """Dataset loader for Yahoo LTR Set2 with click simulation."""
    
    def __init__(self, file_path, scenario=1, device="cpu", K=15, ranked_docs=None):
        self.device = device
        self.scenario = scenario
        self.K = K
        self.ranked_docs = ranked_docs
        
        self.data, self.qids = self.load_yahoo_data(file_path)
        
        self.features = torch.tensor(self.data[:, 2:], dtype=torch.float32).to(self.device)
        self.position = torch.tensor(self.data[:, 0], dtype=torch.float32).unsqueeze(1).to(self.device)
        self.bias_features = torch.tensor(self.data[:, 1], dtype=torch.float32).unsqueeze(1).to(self.device)
        
        feature_index = 50
        theta_values = self.data[:, 2 + feature_index]
        theta_min = np.min(theta_values)
        theta_max = np.max(theta_values)
        
        simulated_clicks = [
            YahooLTRSet2Dataset.simulate_clicks(p, r, theta, self.scenario, theta_min, theta_max)
            for r, p, theta in zip(self.data[:, 1], self.data[:, 0], theta_values)
        ]
        
        self.labels = torch.tensor(simulated_clicks, dtype=torch.float32).unsqueeze(1).to(self.device)
        self.qids = torch.tensor(self.qids, dtype=torch.int64).to(self.device)

    def load_yahoo_data(self, file_path):
        """Load Yahoo LTR Set2 data"""
        data = []
        qid_list = []
        query_groups = {}
        
        print(f"Loading Yahoo LTR Set2 data from {file_path}...")
        
        with open(file_path, 'r') as f:
            for line_num, line in enumerate(f):
                if line_num % 50000 == 0:
                    print(f"Processed {line_num} lines...")
                    
                line = line.strip()
                if not line or line.startswith('#'):
                    continue
                    
                parts = line.split(' ')
                
                relevance = int(parts[0])
                
                qid_part = [part for part in parts if part.startswith('qid:')][0]
                query_id = int(qid_part.split(':')[1])
                
                if query_id not in query_groups:
                    query_groups[query_id] = []
                position = len(query_groups[query_id]) + 1
                
                features = np.zeros(700)
                for item in parts[2:]:
                    if ':' in item and not item.startswith('qid:'):
                        try:
                            index, value = item.split(':')
                            feature_idx = int(index) - 1
                            if 0 <= feature_idx < 700:
                                features[feature_idx] = float(value)
                        except ValueError:
                            continue
                
                query_groups[query_id].append([position, relevance] + list(features))
        
        print(f"Loaded {len(query_groups)} queries from Yahoo LTR Set2")
        
        for qid, docs in query_groups.items():
            if len(docs) <= 5:
                sampled_docs = docs
            else:
                top_5 = docs[:5]
                extra_docs = random.sample(docs[5:], min(self.K, len(docs) - 5))
                sampled_docs = top_5 + extra_docs
            
            for new_pos, doc in enumerate(sampled_docs, start=1):
                doc[0] = float(new_pos)
            
            data.extend(sampled_docs)
            qid_list.extend([qid] * len(sampled_docs))
        
        print(f"Final dataset: {len(data)} documents across {len(query_groups)} queries")
        return np.array(data), np.array(qid_list)

    @staticmethod
    def linear_observation_weight(theta, theta_min, theta_max):
        """Linearly transforms theta to the range [0.5, 1]."""
        normalized = (theta - theta_min) / (theta_max - theta_min + 1e-8)
        return 0.5 + normalized

    @staticmethod
    def simulate_clicks(position, relevance, theta, scenario, theta_min, theta_max):
        """Simulates clicks based on position, relevance, and click simulation scenario."""
        if position <= 0:
            raise ValueError(f"Invalid position: {position}")
        
        binary_relevance = 1 if relevance >= 2 else 0
        
        if scenario <= 4:
            position = max(position, 1)
            observation_prob = 1 / position if position <= 5 else 0.1
        else:
            omega_theta = YahooLTRSet2Dataset.linear_observation_weight(theta, theta_min, theta_max)
            position = max(position, 1)
            observation_prob = omega_theta / position if position <= 5 else 0.1 * omega_theta
        
        observed = np.random.rand() < observation_prob
        
        if scenario == 1:
            click_prob = 1 if binary_relevance == 1 and observed else 0
        elif scenario == 2:
            click_prob = 1 if binary_relevance == 1 and observed else (1 / (min(position, 5) + 3)) if observed else 0
        elif scenario == 3:
            click_prob = 1 if binary_relevance == 1 and observed else (1 / (min(position, 5) + 3)) if observed else (0.1 if binary_relevance == 1 else 0)
        elif scenario == 4:
            click_prob = 1 if binary_relevance == 1 and observed else (1 / (min(position, 5) + 3)) if observed else (0.1 if binary_relevance == 1 else 0.01)
        elif scenario == 5:
            click_prob = 1 if binary_relevance == 1 and observed else (1 / (min(position, 5) + 3)) if observed else (0.1 if binary_relevance == 1 else 0)
        
        click = np.random.rand() < click_prob
        return int(click)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.features[idx], self.bias_features[idx], self.position[idx], self.labels[idx], self.qids[idx]

# ================================
# EVALUATION FUNCTIONS (COPIED FROM YOUR ORIGINAL CODE)
# ================================

def dcg_at_k(r, k):
    """Calculate Discounted Cumulative Gain at rank k"""
    r = np.asarray(r, dtype=float)[:k]
    if r.size:
        return np.sum(r / np.log2(np.arange(2, r.size + 2)))
    return 0.0

def ndcg_at_k(r, k):
    """Calculate Normalized Discounted Cumulative Gain at rank k"""
    dcg_max = dcg_at_k(sorted(r, reverse=True), k)
    if not dcg_max:
        return 0.0
    return dcg_at_k(r, k) / dcg_max

def precision_at_k(r, k):
    """Calculate Precision at rank k"""
    assert k >= 1
    r = np.asarray(r)[:k] != 0
    if r.size != k:
        raise ValueError('Relevance score length < k')
    return np.mean(r)

def average_precision(r):
    """Calculate Average Precision"""
    r = np.asarray(r) != 0
    out = [precision_at_k(r, k + 1) for k, rel in enumerate(r) if rel]
    if not out:
        return 0.0
    return np.mean(out)

def reciprocal_rank(r):
    """Calculate Reciprocal Rank"""
    r = np.asarray(r) != 0
    ranks = np.where(r)[0]
    if len(ranks) > 0:
        return 1.0 / (ranks[0] + 1)
    return 0.0

def evaluate_comprehensive_metrics(model, dataloader, device="cpu"):
    """Comprehensive evaluation with multiple ranking metrics"""
    model.to(device)
    model.eval()
    
    query_predictions = defaultdict(list)
    query_labels = defaultdict(list)
    
    all_preds, all_labels = [], []
    
    with torch.no_grad():
        for x, b, pos, y, qids in dataloader:
            x, b, pos, y = x.to(device), b.to(device), pos.to(device), y.to(device)
            y_pred = model(x, pos)
            
            predictions = y_pred.cpu().numpy().flatten()
            labels = y.cpu().numpy().flatten() 
            qids_np = qids.cpu().numpy()
            
            all_preds.extend(predictions)
            all_labels.extend(labels)
            
            for pred, label, qid in zip(predictions, labels, qids_np):
                query_predictions[qid].append(pred)
                query_labels[qid].append(label)
    
    try:
        auc = roc_auc_score(all_labels, all_preds)
    except ValueError as e:
        print(f"⚠️ AUC calculation failed: {e}")
        auc = 0.0
    
    ndcg_1_scores, ndcg_3_scores, ndcg_5_scores, ndcg_10_scores = [], [], [], []
    map_scores, mrr_scores = [], []
    p_1_scores, p_3_scores, p_5_scores, p_10_scores = [], [], [], []
    
    for qid in query_predictions:
        preds = np.array(query_predictions[qid])
        labels = np.array(query_labels[qid])
        
        if np.sum(labels) == 0:
            continue
            
        sorted_indices = np.argsort(preds)[::-1]
        sorted_labels = labels[sorted_indices]
        
        binary_labels = sorted_labels.astype(int)
        
        if len(sorted_labels) >= 1:
            ndcg_1_scores.append(ndcg_at_k(sorted_labels, 1))
        if len(sorted_labels) >= 3:
            ndcg_3_scores.append(ndcg_at_k(sorted_labels, 3))
        if len(sorted_labels) >= 5:
            ndcg_5_scores.append(ndcg_at_k(sorted_labels, 5))
        if len(sorted_labels) >= 10:
            ndcg_10_scores.append(ndcg_at_k(sorted_labels, 10))
        
        map_scores.append(average_precision(binary_labels))
        mrr_scores.append(reciprocal_rank(binary_labels))
        
        if len(binary_labels) >= 1:
            p_1_scores.append(precision_at_k(binary_labels, 1))
        if len(binary_labels) >= 3:
            p_3_scores.append(precision_at_k(binary_labels, 3))
        if len(binary_labels) >= 5:
            p_5_scores.append(precision_at_k(binary_labels, 5))
        if len(binary_labels) >= 10:
            p_10_scores.append(precision_at_k(binary_labels, 10))
    
    results = {
        'AUC': auc,
        'NDCG@1': np.mean(ndcg_1_scores) if ndcg_1_scores else 0.0,
        'NDCG@3': np.mean(ndcg_3_scores) if ndcg_3_scores else 0.0,
        'NDCG@5': np.mean(ndcg_5_scores) if ndcg_5_scores else 0.0,
        'NDCG@10': np.mean(ndcg_10_scores) if ndcg_10_scores else 0.0,
        'MAP': np.mean(map_scores) if map_scores else 0.0,
        'MRR': np.mean(mrr_scores) if mrr_scores else 0.0,
        'P@1': np.mean(p_1_scores) if p_1_scores else 0.0,
        'P@3': np.mean(p_3_scores) if p_3_scores else 0.0,
        'P@5': np.mean(p_5_scores) if p_5_scores else 0.0,
        'P@10': np.mean(p_10_scores) if p_10_scores else 0.0,
    }
    
    return results

# ================================
# VALIDATION CLASS
# ================================

class YahooLTRValidator:
    """Proper validation framework for Yahoo LTR models"""
    
    def __init__(self, train_file, test_file=None, validation_file=None):
        self.train_file = train_file
        self.test_file = test_file
        self.validation_file = validation_file
        self.results_history = []
        
    def load_separate_datasets(self, device="cpu", scenario=1):
        """Load train, validation, and test sets separately"""
        print("📊 Loading Separate Datasets...")
        
        datasets = {}
        
        if os.path.exists(self.train_file):
            print(f"✅ Loading training data: {self.train_file}")
            datasets['train'] = YahooLTRSet2Dataset(
                self.train_file, scenario=scenario, device=device, K=15
            )
        else:
            raise FileNotFoundError(f"Training file not found: {self.train_file}")
        
        if self.validation_file and os.path.exists(self.validation_file):
            print(f"✅ Loading validation data: {self.validation_file}")
            datasets['val'] = YahooLTRSet2Dataset(
                self.validation_file, scenario=scenario, device=device, K=15
            )
        else:
            print("⚠️  No separate validation file - will split from training data")
            train_size = int(0.8 * len(datasets['train']))
            val_size = len(datasets['train']) - train_size
            datasets['train'], datasets['val'] = torch.utils.data.random_split(
                datasets['train'], [train_size, val_size]
            )
        
        if self.test_file and os.path.exists(self.test_file):
            print(f"✅ Loading test data: {self.test_file}")
            datasets['test'] = YahooLTRSet2Dataset(
                self.test_file, scenario=scenario, device=device, K=15
            )
        else:
            print("⚠️  No separate test file provided")
            datasets['test'] = None
        
        return datasets
    
    def cross_validate_model(self, model_class, datasets, device="cpu", k_folds=3, epochs=70):
        """Perform k-fold cross-validation"""
        print(f"\n🔄 Performing {k_folds}-Fold Cross-Validation...")
        
        if isinstance(datasets['train'], torch.utils.data.dataset.Subset):
            full_dataset = datasets['train'].dataset
        else:
            full_dataset = datasets['train']
        
        all_qids = torch.unique(full_dataset.qids).cpu().numpy()
        np.random.shuffle(all_qids)
        
        fold_results = []
        
        for fold in range(k_folds):
            print(f"\n📁 Fold {fold + 1}/{k_folds}")
            
            fold_size = len(all_qids) // k_folds
            val_start = fold * fold_size
            val_end = val_start + fold_size if fold < k_folds - 1 else len(all_qids)
            
            val_qids = set(all_qids[val_start:val_end])
            train_qids = set(all_qids) - val_qids
            
            train_indices = [i for i, qid in enumerate(full_dataset.qids.cpu().numpy()) if qid in train_qids]
            val_indices = [i for i, qid in enumerate(full_dataset.qids.cpu().numpy()) if qid in val_qids]
            
            train_subset = torch.utils.data.Subset(full_dataset, train_indices)
            val_subset = torch.utils.data.Subset(full_dataset, val_indices)
            
            train_loader = DataLoader(train_subset, batch_size=256, shuffle=True)
            val_loader = DataLoader(val_subset, batch_size=256, shuffle=False)
            
            model = model_class(
                input_dim=700, position_dim=1,
                hidden_dim=32, attention_dim=32
            ).to(device)
            
            optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
            
            print(f"  🏋️ Training fold {fold + 1}...")
            fold_train_losses = self._train_fold(model, train_loader, optimizer, epochs, device)
            
            print(f"  📊 Validating fold {fold + 1}...")
            fold_val_results = evaluate_comprehensive_metrics(model, val_loader, device=device)
            
            fold_results.append({
                'fold': fold + 1,
                'train_losses': fold_train_losses,
                'val_metrics': fold_val_results,
                'train_queries': len(train_qids),
                'val_queries': len(val_qids)
            })
            
            print(f"  ✅ Fold {fold + 1} AUC: {fold_val_results['AUC']:.4f}")
        
        return fold_results
    
    def _train_fold(self, model, train_loader, optimizer, epochs, device):
        """Train model for one fold"""
        criterion = torch.nn.BCELoss()
        model.train()
        
        epoch_losses = []
        
        for epoch in range(epochs):
            epoch_loss = 0
            batch_count = 0
            
            for x, b, pos, y, _ in train_loader:
                x, b, pos, y = x.to(device), b.to(device), pos.to(device), y.to(device)
                
                optimizer.zero_grad()
                y_pred = model(x,  pos)
                loss = criterion(y_pred, y)
                loss.backward()
                optimizer.step()
                
                epoch_loss += loss.item()
                batch_count += 1
            
            avg_loss = epoch_loss / batch_count
            epoch_losses.append(avg_loss)
            
            if epoch % 2 == 0:
                print(f"    Epoch {epoch+1}/{epochs}, Loss: {avg_loss:.4f}")
        
        return epoch_losses
    
    def evaluate_on_holdout_test(self, model, datasets, device="cpu"):
        """Evaluate final model on completely unseen test data"""
        if datasets['test'] is None:
            print("❌ No test dataset available for holdout evaluation")
            return None
        
        print("\n🎯 Final Evaluation on Holdout Test Set...")
        test_loader = DataLoader(datasets['test'], batch_size=256, shuffle=False)
        
        test_results = evaluate_comprehensive_metrics(model, test_loader, device=device)
        
        print("\n" + "="*60)
        print("🏆 HOLDOUT TEST RESULTS (UNBIASED)")
        print("="*60)
        for metric, value in test_results.items():
            print(f"• {metric}: {value:.4f}")
        
        return test_results
    
    def generate_validation_report(self, fold_results, test_results=None):
        """Generate comprehensive validation report"""
        print("\n📊 Generating Validation Report...")
        
        cv_metrics = defaultdict(list)
        for fold in fold_results:
            for metric, value in fold['val_metrics'].items():
                cv_metrics[metric].append(value)
        
        cv_stats = {}
        for metric, values in cv_metrics.items():
            cv_stats[metric] = {
                'mean': np.mean(values),
                'std': np.std(values),
                'min': np.min(values),
                'max': np.max(values),
                'values': values
            }
        
        report = {
            'validation_type': 'k_fold_cross_validation',
            'num_folds': len(fold_results),
            'cross_validation_results': cv_stats,
            'individual_fold_results': fold_results,
            'holdout_test_results': test_results,
            'reliability_assessment': self._assess_reliability(cv_stats, test_results),
            'timestamp': time.strftime('%Y-%m-%d %H:%M:%S')
        }
        
        with open('yahoo_ltr_validation_report.json', 'w') as f:
            json.dump(report, f, indent=2, default=str)
        
        print("\n" + "="*80)
        print("📋 VALIDATION REPORT SUMMARY")
        print("="*80)
        
        print(f"\n🔄 CROSS-VALIDATION RESULTS ({len(fold_results)} folds):")
        for metric, stats in cv_stats.items():
            print(f"• {metric}: {stats['mean']:.4f} ± {stats['std']:.4f} "
                  f"(min: {stats['min']:.4f}, max: {stats['max']:.4f})")
        
        if test_results:
            print(f"\n🎯 HOLDOUT TEST RESULTS:")
            for metric, value in test_results.items():
                print(f"• {metric}: {value:.4f}")
        
        print(f"\n🔍 RELIABILITY ASSESSMENT:")
        assessment = report['reliability_assessment']
        print(f"• Overall Reliability: {assessment['overall_rating']}")
        print(f"• CV Stability: {assessment['cv_stability']}")
        if test_results:
            print(f"• Test Consistency: {assessment['test_consistency']}")
        
        for warning in assessment['warnings']:
            print(f"⚠️  {warning}")
        
        print(f"\n✅ Detailed report saved to yahoo_ltr_validation_report.json")
        
        return report
    
    def _assess_reliability(self, cv_stats, test_results):
        """Assess the reliability of validation results"""
        warnings = []
        
        auc_std = cv_stats['AUC']['std']
        if auc_std > 0.05:
            cv_stability = "Low - High variance across folds"
            warnings.append(f"High AUC variance ({auc_std:.4f}) suggests model instability")
        elif auc_std > 0.02:
            cv_stability = "Medium - Moderate variance"
        else:
            cv_stability = "High - Low variance across folds"
        
        test_consistency = "N/A"
        if test_results:
            cv_auc_mean = cv_stats['AUC']['mean']
            test_auc = test_results['AUC']
            auc_diff = abs(cv_auc_mean - test_auc)
            
            if auc_diff > 0.05:
                test_consistency = "Low - Large gap between CV and test"
                warnings.append(f"Large gap between CV AUC ({cv_auc_mean:.4f}) and test AUC ({test_auc:.4f})")
            elif auc_diff > 0.02:
                test_consistency = "Medium - Some gap between CV and test"
            else:
                test_consistency = "High - Good agreement between CV and test"
        
        if len(warnings) == 0:
            overall_rating = "High - Results are reliable"
        elif len(warnings) <= 2:
            overall_rating = "Medium - Some concerns but generally reliable"
        else:
            overall_rating = "Low - Multiple reliability concerns"
        
        return {
            'overall_rating': overall_rating,
            'cv_stability': cv_stability,
            'test_consistency': test_consistency,
            'warnings': warnings
        }

# ================================
# MAIN VALIDATION RUNNER
# ================================
def run_proper_yahoo_ltr_validation():
    """Run proper validation with train/test splits"""
    
    print("="*80)
    print("🎯 PROPER YAHOO LTR VALIDATION ANALYSIS")
    print("="*80)
    
    # File paths - UPDATE THESE TO YOUR ACTUAL PATHS
    train_file = "C:\\Users\\Amala K J\\Desktop\\yahoo\\set2.train.txt"
    test_file = "C:\\Users\\Amala K J\\Desktop\\yahoo\\set2.test.txt"
    validation_file = None
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"🎯 Using device: {device}")
    
    try:
        validator = YahooLTRValidator(train_file, test_file, validation_file)
        
        datasets = validator.load_separate_datasets(device=device, scenario=1)
        
        print(f"📊 Dataset sizes:")
        print(f"• Training: {len(datasets['train'])} samples")
        print(f"• Validation: {len(datasets['val'])} samples")
        if datasets['test']:
            print(f"• Test: {len(datasets['test'])} samples")
        
        fold_results = validator.cross_validate_model(
            XPA, 
            datasets, 
            device=device, 
            k_folds=3,  # Reduced for faster execution
            epochs=70    # Reduced for faster execution
        )
        
        print("\n🏋️ Training Final Model on Full Training Set...")
        final_model = XPA(
            input_dim=700,  position_dim=1,
            hidden_dim=32, attention_dim=32
        ).to(device)
        
        train_loader = DataLoader(datasets['train'], batch_size=256, shuffle=True)
        optimizer = torch.optim.Adam(final_model.parameters(), lr=0.001)
        
        validator._train_fold(final_model, train_loader, optimizer, epochs=70, device=device)
        
        test_results = None
        if datasets['test']:
            test_results = validator.evaluate_on_holdout_test(final_model, datasets, device=device)
        
        validation_report = validator.generate_validation_report(fold_results, test_results)
        
        return {
            'validation_report': validation_report,
            'final_model': final_model,
            'cv_results': fold_results
        }
        
    except Exception as e:
        print(f"❌ Validation failed: {e}")
        import traceback
        traceback.print_exc()
        return None

# ================================
# EXECUTE PROPER VALIDATION
# ================================
print("🎯 Starting Proper Validation Analysis...")

# Check file paths
train_path = "C:\\Users\\Amala K J\\Desktop\\yahoo\\set2.train.txt"
test_path = "C:\\Users\\Amala K J\\Desktop\\yahoo\\set2.test.txt"

if os.path.exists(test_path):
    print("✅ Found separate test file - will use for unbiased evaluation")
else:
    print("⚠️  No separate test file found - will use cross-validation only")
    print("💡 For production deployment, you should have a separate test set")

# Run validation
print("\n🚀 Running complete validation analysis...")
validation_results = run_proper_yahoo_ltr_validation()

if validation_results:
    print("\n🎉 Proper validation completed!")
    print("📊 Check yahoo_ltr_validation_report.json for detailed results")
    
    # Quick summary of results
    val_report = validation_results['validation_report']
    cv_results = val_report['cross_validation_results']
    
    print("\n" + "="*50)
    print("📋 QUICK VALIDATION SUMMARY")
    print("="*50)
    
    print(f"✅ Cross-Validation AUC: {cv_results['AUC']['mean']:.4f} ± {cv_results['AUC']['std']:.4f}")
    print(f"✅ Cross-Validation NDCG@5: {cv_results['NDCG@5']['mean']:.4f} ± {cv_results['NDCG@5']['std']:.4f}")
    
    if val_report['holdout_test_results']:
        test_auc = val_report['holdout_test_results']['AUC']
        print(f"🎯 Holdout Test AUC: {test_auc:.4f}")
    
    reliability = val_report['reliability_assessment']['overall_rating']
    print(f"🔍 Overall Reliability: {reliability}")
    
    # Determine if results are production-ready
    auc_std = cv_results['AUC']['std']
    auc_mean = cv_results['AUC']['mean']
    
    if auc_std < 0.02 and auc_mean > 0.7:
        print("\n✅ MODEL IS PRODUCTION-READY!")
        print("   • Low variance across folds")
        print("   • Good performance metrics")
        print("   • Reliable for deployment")
    elif auc_std < 0.05:
        print("\n⚠️  MODEL NEEDS MINOR IMPROVEMENTS")
        print("   • Acceptable variance but could be better")
        print("   • Consider more training or hyperparameter tuning")
    else:
        print("\n❌ MODEL NEEDS SIGNIFICANT IMPROVEMENTS")
        print("   • High variance suggests instability")
        print("   • Not recommended for production deployment")

else:
    print("\n❌ Validation failed - check error messages above")
    print("💡 Common issues:")
    print("   • Missing data files")
    print("   • Incorrect file paths") 
    print("   • Memory limitations")


🔹 Using device: cuda
🎯 Starting Proper Validation Analysis...
✅ Found separate test file - will use for unbiased evaluation

🚀 Running complete validation analysis...
🎯 PROPER YAHOO LTR VALIDATION ANALYSIS
🎯 Using device: cuda
📊 Loading Separate Datasets...
✅ Loading training data: C:\Users\Amala K J\Desktop\yahoo\set2.train.txt
Loading Yahoo LTR Set2 data from C:\Users\Amala K J\Desktop\yahoo\set2.train.txt...
Processed 0 lines...
Loaded 1266 queries from Yahoo LTR Set2
Final dataset: 22719 documents across 1266 queries
⚠️  No separate validation file - will split from training data
✅ Loading test data: C:\Users\Amala K J\Desktop\yahoo\set2.test.txt
Loading Yahoo LTR Set2 data from C:\Users\Amala K J\Desktop\yahoo\set2.test.txt...
Processed 0 lines...
Processed 50000 lines...
Processed 100000 lines...
Loaded 3798 queries from Yahoo LTR Set2
Final dataset: 68164 documents across 3798 queries
📊 Dataset sizes:
• Training: 18175 samples
• Validation: 4544 samples
• Test: 68164 samples

🔄 